In [2]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)

Spark version: 4.0.1


# **Bài 3:  Phân Tích Đánh Giá Theo Giới Tính**

In [3]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupations.txt")

In [4]:
movies_rdd.take(5)

['1001,The Godfather (1972),Crime|Drama',
 '1002,The Shawshank Redemption (1994),Drama',
 "1003,Schindler's List (1993),Biography|Drama|History",
 '1004,Raging Bull (1980),Biography|Drama|Sport',
 '1005,Casablanca (1942),Drama|Romance|War']

In [5]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

In [6]:
user_rdd.take(5)

['1,M,28,3,12345',
 '2,F,35,7,23456',
 '3,M,42,2,34567',
 '4,F,19,10,45678',
 '5,M,31,1,56789']

In [8]:
#Mapper lay movieID va title
movie_mapped = movies_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1]))
movie_mapped.take(5)

[('1001', 'The Godfather (1972)'),
 ('1002', 'The Shawshank Redemption (1994)'),
 ('1003', "Schindler's List (1993)"),
 ('1004', 'Raging Bull (1980)'),
 ('1005', 'Casablanca (1942)')]

In [9]:
#Mapper lay rating, userID, movieID
rating_mapped = ratings_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1], x.split(",")[2]))
rating_mapped.take(5)

[('7', '1020', '4.5'),
 ('23', '1015', '3.5'),
 ('45', '1030', '4.0'),
 ('12', '1047', '3.0'),
 ('38', '1012', '4.5')]

In [10]:
#mapper lay userID va gender
user_mapped = user_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1]))
user_mapped.take(5)

[('1', 'M'), ('2', 'F'), ('3', 'M'), ('4', 'F'), ('5', 'M')]

In [ ]:
#Join rating va user theo userID
rating_user_joined = rating_mapped.map(lambda x: (x[0], (x[1], x[2]))).join(user_mapped)
rating_user_joined.take(5)


[('12', (('1047', '3.0'), 'F')),
 ('12', (('1012', '3.5'), 'F')),
 ('12', (('1040', '4.0'), 'F')),
 ('12', (('1013', '4.5'), 'F')),
 ('50', (('1025', '4.5'), 'F'))]

In [12]:
#Map lai lay movieID lam key
rating_user_moviekey = rating_user_joined.map(lambda x: (x[1][0][0], (x[1][0][1], x[1][1])))
rating_user_moviekey.take(5)

[('1047', ('3.0', 'F')),
 ('1012', ('3.5', 'F')),
 ('1040', ('4.0', 'F')),
 ('1013', ('4.5', 'F')),
 ('1025', ('4.5', 'F'))]

In [13]:
#Join rating theo user va moive
rating_user_movie_joined = rating_user_moviekey.join(movie_mapped)
rating_user_movie_joined.take(5)

[('1015', (('4.5', 'F'), 'Sunset Boulevard (1950)')),
 ('1015', (('3.5', 'M'), 'Sunset Boulevard (1950)')),
 ('1015', (('4.5', 'M'), 'Sunset Boulevard (1950)')),
 ('1015', (('4.5', 'M'), 'Sunset Boulevard (1950)')),
 ('1015', (('4.5', 'M'), 'Sunset Boulevard (1950)'))]

In [20]:
#Reduce tinh totalRating va dem so luong rating theo movieTitle va gender
movie_gender_rating = rating_user_movie_joined.map(lambda x: ((x[1][1],x[1][0][1]), (float(x[1][0][0]), 1))) 
movie_gender_rating_reduced = movie_gender_rating.reduceByKey(lambda a,b: (a[0] + b[0], a[1] + b[1]))
#Tinh avg rating
movie_gender_rating_avg = movie_gender_rating_reduced.mapValues(lambda x: x[0] / x[1])
movie_gender_rating_avg.take(10)

[(('Sunset Boulevard (1950)', 'M'), 4.333333333333333),
 (('The Lord of the Rings: The Return of the King (2003)', 'F'), 3.9),
 (('The Terminator (1984)', 'F'), 4.136363636363637),
 (('The Terminator (1984)', 'M'), 3.9285714285714284),
 (('The Lord of the Rings: The Fellowship of the Ring (2001)', 'M'), 4.0),
 (('Fight Club (1999)', 'F'), 3.5),
 (('Gladiator (2000)', 'F'), 3.642857142857143),
 (('Gladiator (2000)', 'M'), 3.590909090909091),
 (('Sunset Boulevard (1950)', 'F'), 4.5),
 (('The Silence of the Lambs (1991)', 'M'), 3.3333333333333335)]

In [ ]:
#Map lai theo movieId
final_result = movie_gender_rating_avg.map(lambda x: (x[0][0], (x[0][1], x[1]))).groupByKey().mapValues(list)
final_result.collect()

[('Mad Max: Fury Road (2015)', [('F', 3.3214285714285716), ('M', 4.0)]),
 ('The Lord of the Rings: The Return of the King (2003)',
  [('F', 3.9), ('M', 3.75)]),
 ('The Terminator (1984)',
  [('F', 4.136363636363637), ('M', 3.9285714285714284)]),
 ('The Silence of the Lambs (1991)', [('M', 3.3333333333333335), ('F', 3.0)]),
 ('No Country for Old Men (2007)',
  [('F', 3.8333333333333335), ('M', 3.9166666666666665)]),
 ('The Godfather: Part II (1974)', [('F', 3.9375), ('M', 4.055555555555555)]),
 ('Lawrence of Arabia (1962)', [('F', 3.3125), ('M', 3.55)]),
 ('The Social Network (2010)', [('F', 3.6666666666666665), ('M', 4.0)]),
 ('Gladiator (2000)', [('F', 3.642857142857143), ('M', 3.590909090909091)]),
 ('Psycho (1960)', [('F', 4.0)]),
 ('Sunset Boulevard (1950)', [('M', 4.333333333333333), ('F', 4.5)]),
 ('The Lord of the Rings: The Fellowship of the Ring (2001)',
  [('M', 4.0), ('F', 3.8)]),
 ('Fight Club (1999)', [('F', 3.5), ('M', 3.5)]),
 ('E.T. the Extra-Terrestrial (1982)', [('M',

In [25]:
#Format result
def format_result(record):
    movie_title = record[0]
    gender_ratings = dict(record[1])
    if "M" in gender_ratings:
        male_avg = f"{gender_ratings['M']:.2f}"
    else:
        male_avg = "NA"
    if "F" in gender_ratings:
        female_avg = f"{gender_ratings['F']:.2f}"
    else:
        female_avg = "NA"
    return f"{movie_title} - Male_Avg: {male_avg}, Female_Avg: {female_avg}"

formatted_result = final_result.map(format_result)
formatted_result.collect()

['Mad Max: Fury Road (2015) - Male_Avg: 4.00, Female_Avg: 3.32',
 'The Lord of the Rings: The Return of the King (2003) - Male_Avg: 3.75, Female_Avg: 3.90',
 'The Terminator (1984) - Male_Avg: 3.93, Female_Avg: 4.14',
 'The Silence of the Lambs (1991) - Male_Avg: 3.33, Female_Avg: 3.00',
 'No Country for Old Men (2007) - Male_Avg: 3.92, Female_Avg: 3.83',
 'The Godfather: Part II (1974) - Male_Avg: 4.06, Female_Avg: 3.94',
 'Lawrence of Arabia (1962) - Male_Avg: 3.55, Female_Avg: 3.31',
 'The Social Network (2010) - Male_Avg: 4.00, Female_Avg: 3.67',
 'Gladiator (2000) - Male_Avg: 3.59, Female_Avg: 3.64',
 'Psycho (1960) - Male_Avg: NA, Female_Avg: 4.00',
 'Sunset Boulevard (1950) - Male_Avg: 4.33, Female_Avg: 4.50',
 'The Lord of the Rings: The Fellowship of the Ring (2001) - Male_Avg: 4.00, Female_Avg: 3.80',
 'Fight Club (1999) - Male_Avg: 3.50, Female_Avg: 3.50',
 'E.T. the Extra-Terrestrial (1982) - Male_Avg: 3.81, Female_Avg: 3.55']